# Stochastic Calculus, Step by Step · Lesson 15: The Itô integral

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Gains from trading: why a new integral · 3. Left, right or middle: it matters · 4. Three properties · 5. The integral of W dW · 6. A deterministic integrand gives a normal · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 15,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. Gains from trading: why a new integral

Hold $H_{t_i}$ shares from $t_i$ to $t_{i+1}$; the gain is $\sum_iH_{t_i}(W_{t_{i+1}} - W_{t_i})$. The holding must be chosen at the **start** of each step, with what is known then. The **Itô integral** $\int_0^TH_t\,dW_t$ is the limit of these sums as the steps shrink. Ordinary (Riemann-Stieltjes) integration needs a path of finite length; Lesson 14 showed Brownian paths are infinitely long, so the limit has to be built differently: in mean square, using the left endpoint.

A first example: hold one share whenever the price is above where it started, nothing otherwise (with $W$ standing in for the price move).

In [ ]:
rng = np.random.default_rng(seed)
fine_n = 2**18
dW_f = rng.normal(0, math.sqrt(1 / fine_n), fine_n)
W_f = np.concatenate([[0], np.cumsum(dW_f)])
grid_f = np.linspace(0, 1, fine_n + 1)
H = (W_f[:-1] > 0).astype(float)                       # decided at the left end of each step
gain = float((H * dW_f).sum())
out["trade"] = {"gain": gain, "W_T": float(W_f[-1]), "share_above": float(H.mean())}
out["trade"]

## 3. Left, right or middle: it matters

For an ordinary smooth path, it doesn't matter where in each step you evaluate the integrand. For Brownian motion it does. Take $\int_0^1W\,dW$ on one path:

- **left** endpoint: $\sum W_{t_i}\Delta W_i \to \tfrac12(W_T^2 - T)$, the **Itô** integral;
- **right** endpoint: $\sum W_{t_{i+1}}\Delta W_i \to \tfrac12(W_T^2 + T)$;
- **average** of the two (Stratonovich): $\to \tfrac12W_T^2$, the ordinary-calculus answer.

Right minus left is exactly $\sum\Delta W_i^2$, the quadratic variation, which tends to $T$, not 0.

In [ ]:
end_rows = []
WT = W_f[-1]
for k_ in (4, 8, 12, 16, 18):
    n = 2**k_
    Wn = W_f[:: fine_n // n]; d_ = np.diff(Wn)
    left, right = float((Wn[:-1] * d_).sum()), float((Wn[1:] * d_).sum())
    end_rows.append({"n": n, "left": left, "right": right, "mid": (left + right) / 2, "gap": right - left})
out["ends"] = {"rows": end_rows, "W_T": float(WT), "ito": 0.5 * (WT**2 - 1), "right_limit": 0.5 * (WT**2 + 1),
               "strat": 0.5 * WT**2}
pd.DataFrame(end_rows)

## 4. Three properties

For an integrand $H$ chosen with past information only (adapted),

1. **Martingale:** $I_t = \int_0^tH\,dW$ is a martingale, so $E[I_T] = 0$;
2. **Itô isometry:** $E\big[I_T^2\big] = E\int_0^TH_t^2\,dt$;
3. **Quadratic variation:** $[I]_T = \int_0^TH_t^2\,dt$, path by path.

Checked on 20,000 paths for $H = W$, where $E\int_0^1W_t^2\,dt = \int_0^1t\,dt = \tfrac12$:

In [ ]:
n_paths, n_steps = 20_000, 1000
dt = 1 / n_steps
dW = rng.normal(0, math.sqrt(dt), (n_paths, n_steps))
W = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(dW, axis=1)], axis=1)
I = (W[:, :-1] * dW).sum(axis=1)                      # Ito integral of W dW, left endpoints
int_H2 = (W[:, :-1] ** 2 * dt).sum(axis=1)            # integral of W^2 dt, path by path
qv_I = ((W[:, :-1] * dW) ** 2).sum(axis=1)            # quadratic variation of the integral
props = {"mean": float(I.mean()), "second": float((I**2).mean()), "iso_rhs": float(int_H2.mean()), "iso_theory": 0.5,
         "qv_gap": float(np.abs(qv_I - int_H2).mean()), "formula_gap": float(np.abs(I - 0.5 * (W[:, -1] ** 2 - 1)).mean())}
out["props"] = props
props

## 5. The integral of W dW

The algebra behind the left-endpoint answer: from $W_{t_{i+1}}^2 - W_{t_i}^2 = 2W_{t_i}\Delta W_i + (\Delta W_i)^2$, summing over $i$ telescopes the left side to $W_T^2$, so

$$\sum_iW_{t_i}\Delta W_i = \tfrac12W_T^2 - \tfrac12\sum_i(\Delta W_i)^2 \;\to\; \tfrac12W_T^2 - \tfrac12T .$$

The extra $-\tfrac12T$ is the first sign of Itô's lemma (Lesson 16). Its distribution: $\tfrac12(W_T^2 - T)$, with mean 0 and variance $\tfrac12T^2$.

In [ ]:
edges = np.linspace(-1, 3, 41)
h, _ = np.histogram(I, bins=edges, density=True)
mid = (edges[1:] + edges[:-1]) / 2
out["wdw"] = {"var": float(I.var()), "var_theory": 0.5, "p_negative": float((I < 0).mean()),
              "p_negative_theory": 2 * N(1) - 1,
              "fig": {"mid": mid.tolist(), "hist": h.tolist(), "x_range": [-1, 3],
                      "ticks": [{"x": x, "label": f"{x:g}"} for x in (-1, 0, 1, 2, 3)]}}
out["wdw"]["p_negative"], out["wdw"]["p_negative_theory"]

## 6. A deterministic integrand gives a normal

If $H_t = f(t)$ is an ordinary function, $\int_0^Tf(t)\,dW_t$ is a sum of independent normal pieces $f(t_i)\Delta W_i$, so it is **normal**, with mean 0 and (by the isometry) variance $\int_0^Tf(t)^2\,dt$. Example: $\int_0^1t\,dW_t \sim N(0, \tfrac13)$.

In [ ]:
tgrid = np.linspace(0, 1, n_steps + 1)[:-1]
J = (tgrid * dW).sum(axis=1)
edges = np.linspace(-2, 2, 41)
hJ, _ = np.histogram(J, bins=edges, density=True)
midJ = (edges[1:] + edges[:-1]) / 2
v = 1 / 3
out["det"] = {"mean": float(J.mean()), "var": float(J.var()), "var_theory": v,
              "p_above_half": float((J > 0.5).mean()), "p_above_half_theory": 1 - N(0.5 / math.sqrt(v)),
              "cov_W1": float(np.mean(J * W[:, -1])), "cov_W1_theory": 0.5,
              "fig": {"mid": midJ.tolist(), "hist": hJ.tolist(),
                      "normal": (np.exp(-midJ**2 / (2 * v)) / math.sqrt(2 * math.pi * v)).tolist(), "x_range": [-2, 2],
                      "ticks": [{"x": x, "label": f"{x:g}"} for x in (-2, -1, 0, 1, 2)]}}
{k_: v_ for k_, v_ in out["det"].items() if k_ != "fig"}

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 3: Solutions*.

**Exercise 1.** Show that $\sum_iW_{t_i}\Delta W_i = \tfrac12W_T^2 - \tfrac12\sum_i(\Delta W_i)^2$ exactly, for any partition, and deduce the limit.

**Exercise 2.** Find the limit of the right-endpoint sum $\sum_iW_{t_{i+1}}\Delta W_i$. Why can't a trader earn the right-endpoint sum?

**Exercise 3.** Use the martingale property and the isometry to find the mean and variance of $\int_0^TW_t\,dW_t$. Check against $\tfrac12(W_T^2 - T)$ directly, using $E[W_T^4] = 3T^2$.

**Exercise 4.** Show that $J = \int_0^1t\,dW_t$ is normal with mean 0 and variance $\tfrac13$, and compute $P(J > 0.5)$.

**Exercise 5.** Compute $\mathrm{Cov}\big(W_1, \int_0^1t\,dW_t\big)$. (Write $W_1 = \int_0^11\,dW_t$; for two integrals, $E[\int f\,dW\int g\,dW] = \int fg\,dt$.)

**Exercise 6.** Show that $\int_0^Tt\,dW_t = TW_T - \int_0^TW_t\,dt$, starting from the discrete identity $t_{i+1}W_{t_{i+1}} - t_iW_{t_i} = t_i\Delta W_i + W_{t_{i+1}}\Delta t$.

**Exercise 7.** (Code) On one Brownian path, compute the left, right and average sums for $\int_0^1W\,dW$ with $2^k$ steps, $k = 4, \dots, 18$, and compare with $\tfrac12(W_1^2 \mp 1)$ and $\tfrac12W_1^2$.

**Exercise 8.** (Code) Simulate 100,000 values of $\int_0^1t\,dW_t$ and of $\int_0^1W_t\,dW_t$. Compare means and variances with the theory, and the probability that each is negative.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")